# 03 · GC10-DET - Transfer Learning + MLflow

10-class classification of steel defects with strong class imbalance.
Key metric: **Macro F1** (not Accuracy!).

| MLflow Experiment | `gc10_steel_defect_classification` |
|---|---|
| Metric | val_f1_macro |
| Server | `SQLite (mlflow.db)` |

In [ ]:
import sys
from pathlib import Path

_nb_root = Path('../..').resolve()
if str(_nb_root) not in sys.path:
    sys.path.insert(0, str(_nb_root))

from utils.arkon_utils import (
    get_device, get_mlflow_uri, save_figure,
    Timer, CheckpointManager, recommended_num_workers
)
print('arkon_utils loaded ✓')

In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from pathlib import Path
from PIL import Image
import numpy as np, matplotlib.pyplot as plt
from sklearn.metrics import classification_report, f1_score, confusion_matrix
import seaborn as sns
import mlflow, mlflow.pytorch

device = get_device()
DEVICE = device
ASSETS = 'cv/gc10'


In [ ]:
import mlflow
import mlflow.pytorch

# MLflow - platform-safe SQLite (works on Windows + Mac)
mlflow.set_tracking_uri(get_mlflow_uri())
EXPERIMENT_NAME = "arkon-cv-gc10"
RUN_NAME        = "resnet18_weighted_v1"

mlflow.set_tracking_uri(MLFLOW_URI)
mlflow.set_experiment(EXPERIMENT_NAME)
print(f'MLflow URI: {mlflow.get_tracking_uri()}')

MLFLOW_TAGS = {"dataset": "gc10_det", "task": "multiclass_classification", "framework": "pytorch", "architecture": "resnet18", "num_classes": "10", "imbalanced": "true", "department": "Arkon_SheetMetal"}
print(f"MLflow URI : {MLFLOW_URI}")
print(f"Experiment : {EXPERIMENT_NAME}")

In [ ]:
DATA_DIR  = Path('../../../data/06_gc10/raw')
MODEL_DIR = Path('../../../models/06_gc10')
MODEL_DIR.mkdir(parents=True, exist_ok=True)
CKPT_DIR  = Path('../../../models/checkpoints/gc10')

CLASS_MAP = {
    '1': 'punching_hole', '2': 'welding_line', '3': 'crescent_gap',
    '4': 'water_spot',    '5': 'oil_spot',      '6': 'silk_spot',
    '7': 'inclusion',     '8': 'rolled_pit',    '9': 'crease', '10': 'waist_folding'
}
NUM_CLASSES = len(CLASS_MAP)

PARAMS = {
    'img_size'   : 224,
    'batch_size' : 64 if device.type == 'cuda' else 32,
    'epochs'     : 30,
    'lr'         : 1e-4,
    'dropout'    : 0.4,
    'num_workers': recommended_num_workers(device),
    'pin_memory' : device.type == 'cuda',
    'seed'       : 42,
}
torch.manual_seed(PARAMS['seed'])
if torch.cuda.is_available(): torch.cuda.manual_seed(PARAMS['seed'])
print(f'Config: {PARAMS}')


## 1. Dataset + WeightedRandomSampler

In [ ]:
class GC10Dataset(Dataset):
    def __init__(self, root, class_map, transform=None, val_split=0.2, split='train', seed=42):
        self.transform = transform; self.classes = list(class_map.values())
        self.class_to_idx = {v: i for i, v in enumerate(self.classes)}
        self.samples = []
        rng = np.random.RandomState(seed)
        for folder_id, cls_name in class_map.items():
            imgs = sorted(list((root/folder_id).glob('*.jpg')) +
                          list((root/folder_id).glob('*.bmp')))
            idx = self.class_to_idx[cls_name]
            n_val = max(1, int(len(imgs)*val_split))
            perm  = rng.permutation(len(imgs))
            chosen = [imgs[i] for i in (perm[n_val:] if split=='train' else perm[:n_val])]
            self.samples.extend([(p, idx) for p in chosen])
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        p, lbl = self.samples[idx]
        img = Image.open(p).convert('RGB')
        if self.transform: img = self.transform(img)
        return img, lbl

SZ   = PARAMS["img_size"]
MEAN = [0.485, 0.456, 0.406]; STD = [0.229, 0.224, 0.225]
train_tf = transforms.Compose([
    transforms.Resize((SZ,SZ)), transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(), transforms.RandomRotation(15),
    transforms.ColorJitter(0.3, 0.3),
    transforms.ToTensor(), transforms.Normalize(MEAN, STD)])
val_tf = transforms.Compose([
    transforms.Resize((SZ,SZ)), transforms.ToTensor(), transforms.Normalize(MEAN, STD)])

train_set = GC10Dataset(DATA_DIR, CLASS_MAP, train_tf, split='train')
val_set   = GC10Dataset(DATA_DIR, CLASS_MAP, val_tf,   split='val')

lbl_counts = Counter([s[1] for s in train_set.samples])
weights    = [1.0/lbl_counts[l] for _,l in train_set.samples]
sampler    = WeightedRandomSampler(weights, len(weights), replacement=True)

BS = PARAMS["batch_size"]
train_loader = DataLoader(train_set, BS, sampler=sampler,   num_workers=0)
val_loader   = DataLoader(val_set,   BS, shuffle=False,     num_workers=0)
print(f"Train: {len(train_set)} | Val: {len(val_set)}")
print(f"Class distribution: {dict(sorted(lbl_counts.items()))} → imbalance ratio: {max(lbl_counts.values())/min(lbl_counts.values()):.1f}x")

## 2. Model + Weighted Loss

In [ ]:
model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
model.fc = nn.Sequential(nn.Dropout(PARAMS["dropout"]),
                         nn.Linear(512, NUM_CLASSES))
model = model.to(DEVICE)

# class weights for the loss - extra protection against imbalance
total_s = sum(lbl_counts.values())
class_w = torch.tensor([total_s/(NUM_CLASSES*lbl_counts.get(i,1))
                         for i in range(NUM_CLASSES)], dtype=torch.float).to(DEVICE)

criterion = nn.CrossEntropyLoss(weight=class_w)
optimizer = torch.optim.Adam(model.parameters(), lr=PARAMS["lr"])
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=PARAMS["epochs"])
print(f"Class weights: {class_w.cpu().numpy().round(3)}")

## 3. Training + MLflow

In [ ]:
scaler = torch.cuda.amp.GradScaler(enabled=(device.type == 'cuda'))

def run_epoch(model, loader, criterion, optimizer, device, train=True):
    model.train(train)
    loss_sum, correct, total = 0, 0, 0
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for X, y in loader:
            X, y = X.to(device), y.to(device)
            with torch.autocast(device_type=device.type, enabled=(device.type == 'cuda')):
                out  = model(X)
                loss = criterion(out, y)
            if train:
                optimizer.zero_grad()
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            loss_sum += loss.item() * len(y)
            correct  += (out.argmax(1) == y).sum().item()
            total    += len(y)
    return loss_sum / total, correct / total

def get_preds_labels(model, loader, device):
    model.eval()
    preds, labels = [], []
    with torch.no_grad():
        for X, y in loader:
            with torch.autocast(device_type=device.type, enabled=(device.type == 'cuda')):
                out = model(X.to(device))
            preds.extend(out.argmax(1).cpu().tolist())
            labels.extend(y.tolist())
    return preds, labels


In [ ]:
CKPT_DIR = Path('../../../models/checkpoints/gc10_resnet18')
ckpt = CheckpointManager(CKPT_DIR)

if ckpt.exists_torch('gc10_resnet18_best'):
    print('\n⚡ Checkpoint found - skipping training')
    state, meta = ckpt.load_torch('gc10_resnet18_best', map_location=device)
    model.load_state_dict(state)
    history = meta.get('history', {})
    train_time_str = meta.get('train_time', 'unknown')
    print(f'  best_epoch   : {meta.get("best_epoch", "?")}')
    print(f'  val_acc      : {meta.get("val_acc", meta.get("val_f1", "?"))}')
    print(f'  train time   : {train_time_str}')
else:
    with Timer('gc10_resnet18 training') as train_timer:
        with mlflow.start_run(run_name=RUN_NAME, tags=MLFLOW_TAGS) as run:
            mlflow.log_params(PARAMS)
            # Log the imbalance as a parameter
            mlflow.log_param("imbalance_ratio",
                             round(max(lbl_counts.values())/min(lbl_counts.values()), 1))
            print(f"Run ID: {run.info.run_id}")
        
            history = {k: [] for k in ['train_loss','val_loss','train_acc','val_acc','val_f1_macro']}
            best_f1 = 0.0
        
            print(f"{'Ep':>3} | {'TrLoss':>8} | {'TrAcc':>7} | {'VlLoss':>8} | {'VlAcc':>7} | {'F1mac':>6}")
            print("-" * 58)
        
            for epoch in range(1, PARAMS["epochs"]+1):
                tl, ta = run_epoch(model, train_loader, criterion, optimizer, DEVICE, train=True)
                vl, va = run_epoch(model, val_loader,   criterion, None,      DEVICE, train=False)
                scheduler.step()
        
                # Macro F1 on val - the key metric under imbalance
                preds_e, labels_e = get_preds_labels(model, val_loader, DEVICE)
                f1_mac = f1_score(labels_e, preds_e, average='macro', zero_division=0)
        
                mlflow.log_metrics({
                    "train_loss" : round(tl, 6), "val_loss"    : round(vl, 6),
                    "train_acc"  : round(ta, 6), "val_acc"     : round(va, 6),
                    "val_f1_macro": round(f1_mac, 6),
                }, step=epoch)
        
                for k, v in zip(['train_loss','val_loss','train_acc','val_acc','val_f1_macro'],
                                [tl, vl, ta, va, f1_mac]):
                    history[k].append(v)
        
                # Checkpoint on F1 (not accuracy!)
                if f1_mac > best_f1:
                    best_f1 = f1_mac
                    torch.save(model.state_dict(), MODEL_DIR / 'resnet18_gc10_best.pth')
        
                print(f"{epoch:>3} | {tl:>8.4f} | {ta:>7.4f} | {vl:>8.4f} | {va:>7.4f} | {f1_mac:>6.4f}")
        
            # ── Training curves ───────────────────────────────────────────────────────
            fig, axes = plt.subplots(1, 3, figsize=(16, 4))
            axes[0].plot(history['train_loss'],label='Train'); axes[0].plot(history['val_loss'],label='Val')
            axes[0].set_title('Loss'); axes[0].legend()
            axes[1].plot(history['train_acc'],label='Train'); axes[1].plot(history['val_acc'],label='Val')
            axes[1].set_title('Accuracy'); axes[1].legend()
            axes[2].plot(history['val_f1_macro'], color='darkorange', linewidth=2)
            axes[2].set_title('Val Macro F1 ← key metric')
            plt.suptitle(f'GC10 ResNet18 | best Macro F1={best_f1:.4f}')
            plt.tight_layout()
            with tempfile.TemporaryDirectory() as tmp:
                p = os.path.join(tmp,"training_curves.png")
                plt.savefig(p, dpi=120); save_figure(fig, 'cv_gc10_plot_1', subfolder='cv/gc10')
plt.show(); mlflow.log_artifact(p, "plots")
        
            # ── Confusion matrix ──────────────────────────────────────────────────────
            model.load_state_dict(torch.load(MODEL_DIR/'resnet18_gc10_best.pth', map_location=DEVICE))
            preds_final, labels_final = get_preds_labels(model, val_loader, DEVICE)
            report = classification_report(labels_final, preds_final, target_names=CLASSES,
                                           output_dict=True, zero_division=0)
            mlflow.log_metrics({
                "best_val_f1_macro"   : round(best_f1, 6),
                "final_val_accuracy"  : round(report["accuracy"], 6),
                "final_f1_weighted"   : round(report["weighted avg"]["f1-score"], 6),
            })
            print(classification_report(labels_final, preds_final, target_names=CLASSES, zero_division=0))
        
            cm = confusion_matrix(labels_final, preds_final)
            fig2, ax = plt.subplots(figsize=(10, 9))
            ConfusionMatrixDisplay(cm, display_labels=[c[:10] for c in CLASSES]).plot(
                cmap='Blues', ax=ax, xticks_rotation=45)
            ax.set_title(f'GC10 ResNet18 - Val Confusion Matrix | F1={best_f1:.4f}')
            plt.tight_layout()
            with tempfile.TemporaryDirectory() as tmp:
                p = os.path.join(tmp,"confusion_matrix.png")
                plt.savefig(p, dpi=120); save_figure(fig, 'cv_gc10_plot_1', subfolder='cv/gc10')
plt.show(); mlflow.log_artifact(p, "plots")
        
            # ── Model registry ────────────────────────────────────────────────────────
            mlflow.pytorch.log_model(model, "model", registered_model_name="gc10_resnet18")
            mlflow.log_artifact(str(MODEL_DIR/'resnet18_gc10_best.pth'), "weights")
            print(f"\n✅ Run complete | best Macro F1={best_f1:.4f}")
    train_time_str = train_timer.report()
    # Save best checkpoint with metadata
    ckpt.save_torch(model.state_dict(), 'gc10_resnet18_best',
                    metadata={'train_time': train_time_str, 'history': history})


## Summary

| Parameter | Value |
|---|---|
| MLflow Experiment | `gc10_steel_defect_classification` |
| Model Registry | `gc10_resnet18` |
| Checkpoint on | **Macro F1** (not Accuracy!) |
| Imbalance handling | WeightedRandomSampler + class_weight in the loss |
| Artifacts | curves (loss/acc/f1), confusion matrix, weights |

In [ ]:
# ── Training curves ──────────────────────────────────────────────
if history:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    ep = range(1, len(history.get('train_loss', [])) + 1)
    axes[0].plot(ep, history['train_loss'], label='Train')
    axes[0].plot(ep, history['val_loss'],   label='Val')
    axes[0].set_title('Loss'); axes[0].legend()
    axes[1].plot(ep, history['train_acc'], label='Train')
    axes[1].plot(ep, history['val_acc'],   label='Val')
    axes[1].set_title('Accuracy'); axes[1].legend()
    plt.suptitle(f'ResNet18 - GC10 10-class  |  {train_time_str}')
    plt.tight_layout()
    save_figure(fig, 'cv_gc10_training_curves', subfolder='cv/gc10')
    plt.show()
